# Prediction: Employee Performance Analysis, INX Future Inc.

This notebook **uses** the models trained in `train_model.ipynb`. It shows how to:
1. Predict the performance rating of a single employee from input factors
2. Predict for several employees at once (a CSV or table)
3. Use the pre-hire model for candidates, with its limits
4. Produce a **watch list** of employees at risk of lower performance, by department (using out-of-fold predictions)

> Reminder: a predicted rating is a **decision-support signal**, not a verdict on a person. Use it to start a supportive conversation and find the cause, not to penalise.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import json, joblib
import numpy as np, pandas as pd
pd.set_option('display.max_columns', 60, 'display.width', 200)

ROOT = Path('../..').resolve()
ART = ROOT/'src'/'models'/'artifacts'
model = joblib.load(ART/'performance_model.joblib')
prehire_model = joblib.load(ART/'prehire_model.joblib')
meta = json.loads((ART/'model_metadata.json').read_text())
print('Selected model     :', meta['metrics']['selected_model'])
print('Top 3 factors      :', meta['top3_factors'])
print('Test macro F1      :', round(meta['metrics']['test_macro_f1'],3), '| test accuracy:', round(meta['metrics']['test_accuracy'],3))

Selected model     : Random Forest
Top 3 factors      : ['EmpEnvironmentSatisfaction', 'EmpLastSalaryHikePercent', 'PromotionDelayRatio']
Test macro F1      : 0.902 | test accuracy: 0.933


## 1. Helper function: predict one or many employees

In [2]:
RATING_LABEL = {2:'2 - Good (lower)', 3:'3 - Excellent', 4:'4 - Outstanding'}

def add_engineered_features(df):
    # Must match the formulas used in data_processing.ipynb. Each feature is only created when its inputs are present
    # (a job candidate, for example, has no "years at this company").
    out = df.copy()
    if {'YearsSinceLastPromotion','ExperienceYearsAtThisCompany'} <= set(out.columns):
        out['PromotionDelayRatio'] = out['YearsSinceLastPromotion'] / (out['ExperienceYearsAtThisCompany'] + 1)
    if {'ExperienceYearsInCurrentRole','ExperienceYearsAtThisCompany'} <= set(out.columns):
        out['RoleStagnationRatio'] = out['ExperienceYearsInCurrentRole'] / (out['ExperienceYearsAtThisCompany'] + 1)
    if {'TotalWorkExperienceInYears','NumCompaniesWorked'} <= set(out.columns):
        out['AvgYearsPerCompany']  = out['TotalWorkExperienceInYears'] / (out['NumCompaniesWorked'] + 1)
    return out

def predict_performance(employees, which='full'):
    """employees: dict (one employee) or DataFrame (many). Returns predicted rating and class probabilities."""
    df = pd.DataFrame([employees]) if isinstance(employees, dict) else employees.copy()
    df = add_engineered_features(df)
    mdl, cols = (model, meta['input_columns']) if which=='full' else (prehire_model, meta['metrics']['prehire_features'])
    missing = [c for c in cols if c not in df.columns]
    if missing: raise ValueError(f'Missing input columns: {missing}')
    proba = mdl.predict_proba(df[cols])
    res = pd.DataFrame(proba, columns=['Prob_Rating2','Prob_Rating3','Prob_Rating4'], index=df.index).round(3)
    res.insert(0, 'PredictedRating', [RATING_LABEL[c] for c in mdl.classes_[proba.argmax(1)]])
    return res

## 2. Predict a single employee

Two example employees with identical profiles except for the three key factors.

In [3]:
base = dict(Age=34, Gender='Male', EducationBackground='Life Sciences', MaritalStatus='Married', EmpDepartment='Sales',
            EmpJobRole='Sales Executive', BusinessTravelFrequency='Travel_Rarely', DistanceFromHome=8, EmpEducationLevel=3,
            EmpEnvironmentSatisfaction=3, EmpHourlyRate=60, EmpJobInvolvement=3, EmpJobLevel=2, EmpJobSatisfaction=3,
            NumCompaniesWorked=2, OverTime='No', EmpLastSalaryHikePercent=14, EmpRelationshipSatisfaction=3,
            TotalWorkExperienceInYears=10, TrainingTimesLastYear=3, EmpWorkLifeBalance=3, ExperienceYearsAtThisCompany=6,
            ExperienceYearsInCurrentRole=4, YearsSinceLastPromotion=1, YearsWithCurrManager=3, Attrition='No')

happy   = {**base, 'EmpEnvironmentSatisfaction':4, 'EmpLastSalaryHikePercent':22, 'YearsSinceLastPromotion':0}
neutral = base
unhappy = {**base, 'EmpEnvironmentSatisfaction':1, 'EmpLastSalaryHikePercent':12, 'YearsSinceLastPromotion':5}

pd.concat({'Satisfied, big hike, just promoted': predict_performance(happy),
           'Typical profile': predict_performance(neutral),
           'Dissatisfied, small hike, long since promotion': predict_performance(unhappy)})

,,PredictedRating,Prob_Rating2,Prob_Rating3,Prob_Rating4
"Satisfied, big hike, just promoted",0,4 - Outstanding,0.00,0.000,1.000
Typical profile,0,3 - Excellent,0.00,0.997,0.003
"Dissatisfied, small hike, long since promotion",0,2 - Good (lower),0.99,0.010,0.000


## 3. Predict many employees at once

In [4]:
sample = pd.read_csv(ROOT/'data'/'processed'/'employee_clean.csv').sample(8, random_state=7)
out = predict_performance(sample)
out.insert(0, 'EmpNumber', sample['EmpNumber']); out.insert(1, 'ActualRating', sample['PerformanceRating'])
out

,EmpNumber,ActualRating,PredictedRating,Prob_Rating2,Prob_Rating3,Prob_Rating4
459,E1001828,4,4 - Outstanding,0.000,0.000,1.000
457,E1001826,2,2 - Good (lower),0.947,0.017,0.037
1022,E100701,3,3 - Excellent,0.000,0.987,0.013
141,E1001264,3,3 - Excellent,0.027,0.920,0.053
668,E1002196,3,3 - Excellent,0.000,0.990,0.010
1011,E100683,3,3 - Excellent,0.000,0.957,0.043
151,E1001277,3,3 - Excellent,0.010,0.713,0.277
830,E100382,3,3 - Excellent,0.007,0.993,0.000


*(These sample employees were part of the training data, so a close match is expected. For an unbiased view of accuracy, use the test-set results in `train_model.ipynb`.)*

## 4. Pre-hire model for candidates

Only candidate-knowable information is needed. **Please read the limits:** in testing, this model was only slightly better than chance (see `train_model.ipynb`, section 6), so it must not be used as a screening rule.

In [5]:
candidate = {k: base[k] for k in meta['metrics']['prehire_features'] if k in base}   # AvgYearsPerCompany is computed automatically
print(predict_performance(candidate, which='prehire'))
print()
m = meta['metrics']
print(f"Pre-hire model on test data: accuracy {m['prehire_test_accuracy']:.2f}, macro F1 {m['prehire_test_macro_f1']:.2f}, "
      f"balanced accuracy {m['prehire_test_balanced_accuracy']:.2f}  (a coin flip between 3 classes would give 0.33 balanced accuracy)")

   PredictedRating  Prob_Rating2  Prob_Rating3  Prob_Rating4
0  4 - Outstanding         0.336         0.269         0.395

Pre-hire model on test data: accuracy 0.50, macro F1 0.33, balanced accuracy 0.37  (a coin flip between 3 classes would give 0.33 balanced accuracy)


## 5. Watch list: employees at risk of lower performance

Predictions below come from models that **did not see** the employee being predicted (out-of-fold). An employee is placed on the watch list when the model gives a probability of at least 50% to rating 2. The goal is **support and investigation**: managers should find out *why* satisfaction is low and what would help.

In [6]:
oof = pd.read_csv(ART/'out_of_fold_predictions.csv')
src = pd.read_csv(ROOT/'data'/'processed'/'employee_clean.csv')[['EmpNumber','EmpEnvironmentSatisfaction','EmpLastSalaryHikePercent','YearsSinceLastPromotion','EmpWorkLifeBalance']]
oof = oof.merge(src, on='EmpNumber')
oof['OnWatchList'] = oof['Prob_Rating2'] >= 0.5

summary = oof.groupby('EmpDepartment').agg(employees=('EmpNumber','size'), on_watch_list=('OnWatchList','sum'),
              actual_rating2=('ActualRating', lambda s:(s==2).sum()))
summary['watch_list_%'] = (100*summary['on_watch_list']/summary['employees']).round(1)
summary.sort_values('watch_list_%', ascending=False)

,employees,on_watch_list,actual_rating2,watch_list_%
EmpDepartment,,,,
Finance,49,15,15,30.6
Sales,373,96,87,25.7
Research & Development,343,73,68,21.3
Human Resources,54,10,10,18.5
Data Science,20,2,1,10.0
Development,361,19,13,5.3


In [7]:
w = oof[oof.OnWatchList]
print(f'Watch list: {len(w)} employees = {100*len(w)/len(oof):.1f}% of the workforce')
print(f'Of the watch list, {100*(w.ActualRating==2).mean():.0f}% really are rating 2 (precision)')
print(f'The watch list captures {100*((oof.OnWatchList)&(oof.ActualRating==2)).sum()/(oof.ActualRating==2).sum():.0f}% of all rating-2 employees (recall)')
w.sort_values('Prob_Rating2', ascending=False).head(15)[['EmpNumber','EmpDepartment','EmpJobRole','Prob_Rating2','EmpEnvironmentSatisfaction','YearsSinceLastPromotion','EmpLastSalaryHikePercent','ActualRating']]

Watch list: 215 employees = 17.9% of the workforce
Of the watch list, 81% really are rating 2 (precision)
The watch list captures 90% of all rating-2 employees (recall)


,EmpNumber,EmpDepartment,EmpJobRole,Prob_Rating2,EmpEnvironmentSatisfaction,YearsSinceLastPromotion,EmpLastSalaryHikePercent,ActualRating
91,E1001167,Sales,Sales Executive,1.0000,2,5,14,3
550,E1001979,Research & Development,Research Scientist,1.0000,2,8,12,2
611,E1002091,Research & Development,Research Scientist,0.9967,1,1,13,2
737,E1002302,Sales,Sales Executive,0.9967,1,3,19,2
712,E1002264,Sales,Sales Executive,0.9967,2,5,14,2
759,E1002335,Research & Development,Manufacturing Director,0.9967,2,1,19,2
452,E1001821,Research & Development,Research Scientist,0.9933,1,3,17,2
380,E1001685,Research & Development,Research Scientist,0.9933,1,1,12,3
462,E1001831,Research & Development,Healthcare Representative,0.9933,1,7,12,2
618,E1002107,Research & Development,Healthcare Representative,0.9933,2,7,12,2


In [8]:
# Save the full watch list for HR (employee numbers only; no personal attributes)
w.sort_values('Prob_Rating2', ascending=False).to_csv(ROOT/'data'/'processed'/'watch_list.csv', index=False)
print('Saved data/processed/watch_list.csv with', len(w), 'rows')

Saved data/processed/watch_list.csv with 215 rows


**How to use this responsibly**
- The list is for **supportive action** (workload, environment, career conversation), not penalties.
- It is built on workplace factors only, never on gender, age or marital status.
- Because the model is not perfect, every flagged case needs a human conversation before any decision.